# 01 · Wetter: Vorhersagen, Beobachtungen und Schätzungen

Führe die Codezellen mit **Shift + Enter** aus. Lies jeweils zuerst die Frage, probiere dann das Werkzeug aus und besprecht kurz eure Beobachtung. Notizen kannst du direkt bei deinen Papieraufgaben machen. Speichere deine Codeänderungen mit **Strg + S**.

Unser Wettermodell kennt **Sonne (S)** und **Regen (R)**. Dabei sind $a=P(S\text{ morgen}\mid S\text{ heute})$ und $b=P(S\text{ morgen}\mid R\text{ heute})$. In der Reihenfolge S, R lautet die Übergangsmatrix

$$M=\begin{pmatrix}a&b\\1-a&1-b\end{pmatrix}.$$

Jede **Spalte** gehört zum Wetter heute, jede **Zeile** zum Wetter morgen. Das Ausgangsmodell verwendet **a = 80 %, b = 30 %**.

**Schwierigkeitsstufen**

| Stufe | Orientierung |
|---|---|
| Leicht | Ab Klasse 9: Wahrscheinlichkeiten, Baumdiagramme, Bruchrechnen. Oft genügen Ausprobieren und genaues Lesen. |
| Schwerer | Ab Klasse 10/11: Matrizen, Gleichungssysteme, mehrstufige Überlegungen und Argumentieren mit Begriffen aus dem Skript. |
| Schwer | Eher Klasse 12/13: Knobelaufgaben, anspruchsvollere Begründungen, Beweisideen und ungewohnte Modellierung. |

Die Klassenstufen sind eine Orientierung, keine Zugangsvoraussetzung. Du kannst schwierigere Teilaufgaben auch überspringen, die Reihenfolge ist nicht obligatorisch.


<br>

## Vorbereitung der Widgets
#### Klicke zuerst oben im Menü auf **Run → Run All Cells**. Den Code in der hier folgenden Zelle brauchst du nicht zu bearbeiten.

In [1]:
# Pakete und Widgets vorbereiten.
import sys

try:
    import ipywidgets as W
except ModuleNotFoundError:
    if sys.platform == "emscripten":
        import piplite

        await piplite.install("ipywidgets>=8,<9")
        import ipywidgets as W
    else:
        raise RuntimeError("Bitte ipywidgets in dieser Python-Umgebung installieren lassen.")


import io
import html
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from IPython.display import display

PANELS = {}


# Wettermodell und gemeinsame Widget-Hilfen.
def wetter_matrix(a=0.8, b=0.3):
    return np.array([[a, b], [1 - a, 1 - b]], dtype=float)


def beschriftet(text, widget):
    return W.VBox([W.HTML(html.escape(text)), widget])


def regler(wert, maximum=100, minimum=0):
    return W.IntSlider(
        value=wert,
        min=minimum,
        max=maximum,
        continuous_update=True,
        layout=W.Layout(width="480px", max_width="100%"),
    )


def bild_aktualisieren(fig, widget):
    # Dasselbe Bild-Widget austauschen: kein leeres Output-Feld beim Schieben.
    with io.BytesIO() as buffer:
        fig.savefig(buffer, format="png", dpi=105)
        widget.value = buffer.getvalue()


def verlauf(M, p0, tage):
    ps = [np.array(p0, dtype=float)]
    for _ in range(tage):
        ps.append(M @ ps[-1])
    return np.array(ps)


# Stationen.
def station_modelllabor():
    a, b, tage = regler(80), regler(30), regler(20, 100, 1)
    bild = W.Image(format="png", layout=W.Layout(width="700px", max_width="100%"))
    matrix = W.HTML()
    powertext = W.HTML()
    powerbox = W.Accordion(children=[powertext], selected_index=None)
    powerbox.set_title(0, "M^k für beide Startzustände (aufklappen)")
    info = W.HTML()
    fig, ax = plt.subplots(figsize=(6.7, 3.0))
    linien = [
        ax.plot([], [], "o-", ms=3, color="#ad6500", label="Start: sicher Sonne")[0],
        ax.plot([], [], "s--", ms=3, color="#2875aa", label="Start: sicher Regen")[0],
    ]
    ax.set(xlabel="Tag k (Start: Tag 0)", ylabel="P(Sonne am Tag k)", ylim=(-0.04, 1.04))
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=6))
    ax.grid(alpha=0.2)
    ax.legend(loc="upper right")
    fig.tight_layout()
    plt.close(fig)
    record = {
        "a": a,
        "b": b,
        "tage": tage,
        "bild": bild,
        "fig": fig,
        "last": None,
        "updates": 0,
    }

    def update(_=None):
        M = wetter_matrix(a.value / 100, b.value / 100)
        ps = [verlauf(M, [1, 0], tage.value), verlauf(M, [0, 1], tage.value)]
        for line, p in zip(linien, ps):
            line.set_data(np.arange(tage.value + 1), p[:, 0])
        ax.set_xlim(0, tage.value)
        matrix.value = (
            "<b>Matrix (Spalten: heute; Zeilen: morgen; Reihenfolge S, R):</b>"
            '<table style="text-align:center"><tr>'
            + "".join(f'<td style="padding:2px 14px">{v:.2f}</td>' for v in M[0])
            + "</tr><tr>"
            + "".join(f'<td style="padding:2px 14px">{v:.2f}</td>' for v in M[1])
            + "</tr></table>"
        )
        info.value = (
            f"Am Tag {tage.value}: P(Sonne) = <b>{ps[0][-1,0]:.3f}</b> ab Sonne; "
            f"<b>{ps[1][-1,0]:.3f}</b> ab Regen."
        )
        bild_aktualisieren(fig, bild)
        power = np.linalg.matrix_power(M, tage.value)
        powertext.value = tabellen_html(
            ["Ziel ↓ / Start →", "S", "R"],
            [[label] + [f"{v:.6f}" for v in row] for label, row in zip(["S", "R"], power)],
        )
        record["last"] = {"matrix": M, "power": power, "distributions": np.array(ps)}
        record["updates"] += 1

    for w in (a, b, tage):
        w.observe(update, names="value")
    display(
        W.VBox(
            [
                beschriftet("Nach Sonne wieder Sonne: a in %", a),
                beschriftet("Nach Regen Sonne: b in %", b),
                beschriftet("Anzahl Tage", tage),
                matrix,
                bild,
                info,
                powerbox,
            ]
        )
    )
    update()
    record["update"] = update
    PANELS["labor"] = record
    return record


def wetter_ensemble(seed, start, anzahl=1000, tage=60):
    # Fixe Maximalgröße hält beim Verschieben beider Regler alle alten Daten fest.
    zufall = np.random.default_rng(seed).random((anzahl, tage))
    paths = np.empty((anzahl, tage + 1), dtype=np.int8)
    paths[:, 0] = start
    for t in range(tage):
        p_sonne = np.where(paths[:, t] == 0, 0.8, 0.3)
        paths[:, t + 1] = np.where(zufall[:, t] < p_sonne, 0, 1)
    return paths


def station_wetterverlauf():
    start = W.Dropdown(options=[("Sonne", 0), ("Regen", 1)], value=0, description="Start:")
    tage = regler(10, 60, 1)
    anzahl = W.SelectionSlider(
        options=[1, 10, 30, 100, 300, 1000],
        value=1,
        continuous_update=True,
        layout=W.Layout(width="480px", max_width="100%"),
    )
    neu = W.Button(
        description="Neue Versuchsreihe", button_style="info", layout=W.Layout(width="230px")
    )
    bild = W.Image(format="png", layout=W.Layout(width="750px", max_width="100%"))
    info = W.HTML()
    fig, axes = plt.subplots(2, 1, figsize=(7.1, 4.7), sharex=True)
    (probline,) = axes[0].plot(
        [], [], "-", lw=2, color="#ad6500", label="Berechnete Wahrscheinlichkeit"
    )
    (freqline,) = axes[0].plot(
        [], [], "o--", ms=3, color="#2875aa", label="Anteil in unabhängigen Verläufen"
    )
    (pathline,) = axes[1].plot([], [], "o-", ms=3, color="#2875aa", drawstyle="steps-post")
    axes[0].set(
        ylabel="Sonnenanteil /\nP(Sonne)",
        ylim=(-0.05, 1.05),
        title="Wie oft ist am jeweiligen Tag Sonne?",
    )
    axes[0].legend(loc="upper right", fontsize=8)
    axes[1].set(
        ylabel="Wetter",
        yticks=[0, 1],
        yticklabels=["Regen", "Sonne"],
        ylim=(-0.2, 1.2),
        xlabel="Tag k (Start: Tag 0)",
        title="Der erste Verlauf dieser Versuchsreihe",
    )
    axes[1].xaxis.set_major_locator(MaxNLocator(integer=True, nbins=8))
    for ax in axes:
        ax.grid(alpha=0.2)
    fig.tight_layout()
    plt.close(fig)
    r = {
        "start": start,
        "tage": tage,
        "anzahl": anzahl,
        "button": neu,
        "bild": bild,
        "fig": fig,
        "seed": 2030,
        "cache_key": None,
        "last": None,
    }

    def update(_=None):
        key = (r["seed"], start.value)
        if r["cache_key"] != key:
            r["paths"] = wetter_ensemble(*key)
            r["cache_key"] = key
        k, R = tage.value, anzahl.value
        paths = r["paths"][:R, : k + 1]
        ps = verlauf(wetter_matrix(), np.eye(2)[:, start.value], k)
        freq = (paths == 0).mean(axis=0)
        count = int((paths[:, -1] == 0).sum())
        days = np.arange(k + 1)
        probline.set_data(days, ps[:, 0])
        freqline.set_data(days, freq)
        pathline.set_data(days, 1 - paths[0])
        axes[1].set_xlim(0, k)
        bild_aktualisieren(fig, bild)
        info.value = (
            f"<b>Tag {k}:</b> Sonne in {count} von {R} unabhängigen Verläufen "
            f"(<b>{freq[-1]:.1%}</b>); berechnet: <b>{ps[-1,0]:.1%}</b>."
            "<br>Jeder blaue Punkt zählt über die Verläufe am selben Tag, nicht über die Tage eines Verlaufs."
        )
        r["last"] = {
            "path": paths[0].copy(),
            "paths": paths.copy(),
            "distributions": ps,
            "frequency": freq,
            "sun_count": count,
            "trials": R,
            "day": k,
        }

    def new(_):
        r["seed"] += 1
        update()

    for w in (start, tage, anzahl):
        w.observe(update, names="value")
    neu.on_click(new)
    display(
        W.VBox(
            [
                start,
                beschriftet("Bis zu welchem Tag k?", tage),
                beschriftet("Anzahl unabhängiger Wetterverläufe R", anzahl),
                neu,
                bild,
                info,
            ]
        )
    )
    update()
    r["update"] = update
    PANELS["pfad"] = r
    return r


def grosse_stichprobe(seed, start, n=100000):
    zufall = np.random.default_rng(seed).random(n)
    x = np.empty(n + 1, dtype=np.int8)
    x[0] = start
    for t, u in enumerate(zufall, 1):
        x[t] = 0 if u < (0.8 if x[t - 1] == 0 else 0.3) else 1
    return x


def datenregler(wert=100):
    return W.SelectionSlider(
        options=[10, 30, 100, 300, 1000, 3000, 10000, 30000, 100000],
        value=wert,
        continuous_update=True,
        layout=W.Layout(width="480px", max_width="100%"),
    )


def station_zeitanteile():
    n = datenregler()
    start = W.Dropdown(options=[("Sonne", 0), ("Regen", 1)], value=0, description="Start:")
    neu = W.Button(
        description="Neuen Verlauf auslosen",
        button_style="info",
        layout=W.Layout(width="230px"),
    )
    bild = W.Image(format="png", layout=W.Layout(width="700px", max_width="100%"))
    info = W.HTML()
    fig, ax = plt.subplots(figsize=(6.7, 3.1))
    (line,) = ax.plot([], [], color="#ad6500")
    ax.set(
        xscale="log",
        xlim=(1, 100),
        ylim=(-0.02, 1.02),
        xlabel="Gezählte Tage n (logarithmische Achse)",
        ylabel="Bisheriger Sonnenanteil",
    )
    ax.grid(alpha=0.2)
    fig.tight_layout()
    plt.close(fig)
    r = {
        "n": n,
        "start": start,
        "button": neu,
        "bild": bild,
        "seed": 2026,
        "cache_key": None,
        "last": None,
    }

    def update(_=None):
        key = (r["seed"], start.value)
        if r["cache_key"] != key:
            r["path"] = grosse_stichprobe(*key)
            r["sums"] = np.cumsum(r["path"][1:] == 0)
            r["cache_key"] = key
        N = n.value
        idx = np.unique(np.geomspace(1, N, min(N, 600)).astype(int))
        line.set_data(idx, r["sums"][idx - 1] / idx)
        ax.set_xlim(1, N)
        bild_aktualisieren(fig, bild)
        count = int(r["sums"][N - 1])
        freq = count / N
        info.value = (
            f"<b>Sonne: {count} / {N} = {freq:.4f}</b>; "
            f"<b>Regen: {N-count} / {N} = {1-freq:.4f}</b>. "
            "Anteile unter Tagen 1 bis N; Tag 0 wird nicht mitgezählt."
        )
        r["last"] = {"path": r["path"][: N + 1].copy(), "count": count, "frequency": freq}

    def new(_):
        r["seed"] += 1
        update()

    for w in (n, start):
        w.observe(update, names="value")
    neu.on_click(new)
    display(W.VBox([start, beschriftet("Anzahl gezählter Tage N", n), neu, bild, info]))
    update()
    r["update"] = update
    PANELS["zeit"] = r
    return r


def tabellen_html(kopf, rows):
    head = "".join(f'<th style="padding:4px 10px">{html.escape(str(x))}</th>' for x in kopf)
    body = "".join(
        "<tr>"
        + "".join(
            f'<td style="padding:4px 10px;text-align:center">{html.escape(str(x))}</td>'
            for x in row
        )
        + "</tr>"
        for row in rows
    )
    return '<div style="overflow-x:auto"><table><tr>' + head + "</tr>" + body + "</table></div>"


def station_schaetzlabor():
    n = datenregler(100)
    neu = W.Button(
        description="Neue Daten auslosen", button_style="info", layout=W.Layout(width="210px")
    )
    weiter = W.Button(
        description="Einen weiteren Tag aufdecken", layout=W.Layout(width="250px")
    )
    bild = W.Image(format="png", layout=W.Layout(width="760px", max_width="100%"))
    table, info = W.HTML(), W.HTML()
    fig, axes = plt.subplots(2, 1, figsize=(7.2, 6.0))
    colors = ["#ad6500", "#2875aa"]
    estimate_lines = [
        axes[0].plot([], [], color=c, lw=1.4, label=label)[0]
        for c, label in zip(colors, [r"Schätzung $\hat a_n$", r"Schätzung $\hat b_n$"])
    ]
    for value, c, label in zip([0.8, 0.3], colors, ["Modell: a = 0,8", "Modell: b = 0,3"]):
        axes[0].axhline(value, color=c, ls="--", lw=1, label=label)
    error_lines = [
        axes[1].plot([], [], color=c, lw=1.2, label=label)[0]
        for c, label in zip(colors, [r"$100(\hat a_n-a)$", r"$100(\hat b_n-b)$"])
    ]
    axes[1].axhline(0, color="#444444", lw=1, ls="--")
    axes[0].set(
        xscale="log",
        ylim=(-0.04, 1.04),
        ylabel="Geschätzte Wahrscheinlichkeit",
        xlabel="Anzahl Übergänge n",
        title="Schätzungen aus den ersten n Übergängen",
    )
    axes[1].set(xscale="log", ylabel="Fehler in Prozentpunkten", xlabel="Anzahl Übergänge n")
    axes[0].legend(
        loc="lower center", bbox_to_anchor=(0.5, 1.01), fontsize=8, ncol=2, borderaxespad=0
    )
    axes[0].set_title("Schätzungen aus den ersten n Übergängen", pad=43)
    axes[1].legend(loc="upper right", fontsize=8, ncol=2)
    for ax in axes:
        ax.set_xlim(1, 100)
        ax.grid(alpha=0.2)
    fig.tight_layout()
    plt.close(fig)
    r = {
        "n": n,
        "button": neu,
        "next_button": weiter,
        "bild": bild,
        "table": table,
        "info": info,
        "fig": fig,
        "seed": 2026,
        "cache_key": None,
        "last": None,
        "lock": False,
    }

    def update(_=None):
        if r["lock"]:
            return
        if r["cache_key"] != r["seed"]:
            r["path"] = grosse_stichprobe(r["seed"], 0)
            old, new = r["path"][:-1], r["path"][1:]
            r["counts_cum"] = np.array(
                [np.cumsum((old == j) & (new == i)) for j in range(2) for i in range(2)]
            )
            den_all = np.array(
                [r["counts_cum"][:2].sum(axis=0), r["counts_cum"][2:].sum(axis=0)]
            )
            r["estimates"] = np.divide(
                r["counts_cum"][[0, 2]],
                den_all,
                out=np.full(den_all.shape, np.nan),
                where=den_all > 0,
            )
            r["cache_key"] = r["seed"]
        N = n.value
        count = r["counts_cum"][:, N - 1]
        den = np.repeat([count[:2].sum(), count[2:].sum()], 2)
        est = np.divide(count, den, out=np.full(4, np.nan), where=den > 0)
        xs = np.arange(1, N + 1)
        # Show every prefix, including short reversals; no averaging over runs.
        for line, values in zip(estimate_lines, r["estimates"]):
            line.set_data(xs, values[:N])
        lower = max(1, int(np.ceil(N / 10)))
        zoom_x = np.arange(lower, N + 1)
        errors = 100 * (r["estimates"][:, lower - 1 : N] - np.array([0.8, 0.3])[:, None])
        for line, values in zip(error_lines, errors):
            line.set_data(zoom_x, values)
        finite = np.abs(errors[np.isfinite(errors)])
        scale = max(float(finite.max()) * 1.15, 0.01) if finite.size else 1.0
        axes[0].set_xlim(1, max(2, N))
        axes[1].set_xlim(lower, max(lower + 1, N))
        axes[1].set_ylim(-scale, scale)
        axes[1].set_title(
            f"Fehler für n = {lower:,} bis {N:,} · y-Achse automatisch vergrößert".replace(
                ",", " "
            ),
            fontsize=10,
        )
        fig.tight_layout()
        bild_aktualisieren(fig, bild)
        rows = [
            [name, int(c), int(d), "nicht schätzbar" if np.isnan(v) else f"{v:.5f}"]
            for name, c, d, v in zip(["S → S", "S → R", "R → S", "R → R"], count, den, est)
        ]
        table.value = tabellen_html(
            ["Übergang", "Anzahl", "Ausgänge aus Startzustand", "Schätzung bei N"], rows
        )
        names = ["Sonne", "Regen"]
        last_state = int(r["path"][N])
        previous = int(r["path"][N - 1])
        info.value = (
            f"<b>N = {N}:</b> Letzter beobachteter Tag: <b>{names[last_state]}</b> "
            f"(letzter Übergang: {names[previous]} → {names[last_state]}).<br>"
            "Der obere Plot zeigt alle bisherigen Schätzungen. Unten wird nur der letzte Größenbereich "
            "vergrößert; positive Fehler bedeuten Überschätzung, negative Unterschätzung."
        )
        if np.isnan(est).any():
            info.value += "<br>Ein Startzustand kam noch nicht vor: Seine Schätzung fehlt auch in den Kurven."
        weiter.disabled = N >= 100000
        r["last"] = {
            "counts": count.reshape(2, 2).T.copy(),
            "estimate": est.reshape(2, 2).T.copy(),
            "path": r["path"][: N + 1].copy(),
            "last_state": last_state,
            "n": N,
            "zoom_n": zoom_x,
            "errors_pp": errors.copy(),
        }

    def new(_):
        r["seed"] += 1
        update()

    def next_day(_):
        if n.value >= 100000:
            return
        value = n.value + 1
        # Preserve the current selection when adding the next integer to the live slider.
        r["lock"] = True
        try:
            n.options = tuple(sorted(set(n.options) | {value}))
            n.value = value
        finally:
            r["lock"] = False
        update()

    n.observe(update, names="value")
    neu.on_click(new)
    weiter.on_click(next_day)
    display(
        W.VBox(
            [
                beschriftet("Beobachtete Übergänge N (Start: Sonne)", n),
                W.HBox([neu, weiter], layout=W.Layout(flex_flow="row wrap")),
                bild,
                info,
                table,
            ]
        )
    )
    update()
    r["update"] = update
    PANELS["schaetzung"] = r
    return r


print("Bereit – weiter mit Station 1.")


Bereit – weiter mit Station 1.


<br><br>

## 1 · Wie stark wirkt das heutige Wetter nach? (Skriptaufgabe 1.37.1)

Wir untersuchen $p^{(k)}=\big(P(S\text{ am Tag }k),P(R\text{ am Tag }k)\big)^\top$. Die Zelle berechnet wiederholt $p^{(k+1)}=Mp^{(k)}$. Beide Kurven zeigen die **Sonnenwahrscheinlichkeit** für unterschiedliche Starts, keine ausgelosten Wetterverläufe.

1. **Vermute zuerst:** Werden sich die Wahrscheinlichkeiten bei den Starts „sicher Sonne“ und „sicher Regen“ mit der Zeit annähern? Lass zunächst **a = 80 %, b = 30 %** stehen und vergleiche die Kurven bei **k = 2, 10, 50 und 100**. Was erwartest du für noch größere k? <span style="color: #707070;">(leicht)</span>
2. Verändere a oder b. Was verändert sich an den Kurven? <span style="color: #707070;">(leicht)</span>
3. Klappe bei **a = 80 %, b = 30 %** die Matrizen $M^k$ für **k = 2, 10, 50 und 100** auf. Erkläre den Zusammenhang zwischen den Spalten und den beiden Kurven. Was folgt aus den fast gleichen Spalten von $M^{100}$ für die Abhängigkeit von $p^{(100)}=M^{100}p$ von der Anfangsverteilung p? <span style="color: #707070;">(schwerer)</span>
4. Untersuche **a = 100 %, b = 0 %** und **a = 0 %, b = 100 %**. Erkläre anhand der Übergangsregeln, warum hier keine vom Startzustand unabhängige Konvergenz eintritt. <span style="color: #707070;">(schwerer)</span>
5. Für **0 < a, b < 1** konvergiert die Sonnenwahrscheinlichkeit gegen ein festes Level, unabhängig vom Startzustand. Berechne diese Grenzwahrscheinlichkeit in Abhängigkeit von a und b. <span style="color: #707070;">(schwerer)</span>

Dabei ist $a=P(S\text{ morgen}\mid S\text{ heute})$ und $b=P(S\text{ morgen}\mid R\text{ heute})$.


In [2]:
station_1 = station_modelllabor()

<br><br>

## 2 · Wie lässt sich eine Wettervorhersage überprüfen?

Hier gilt wieder das **feste Ausgangsmodell mit a = 80 %, b = 30 %**, unabhängig von den Reglern in Station 1.

Die orange Kurve ist die berechnete Sonnenwahrscheinlichkeit. Die blaue Kurve zählt an jedem Tag den Sonnenanteil über **R unabhängige Wetterverläufe mit demselben Start**. Darunter siehst du den ersten dieser Verläufe.

1. Stelle **Start Sonne, k = 10, R = 1** ein. Angenommen, am Tag 10 regnet es: Widerlegt das eine Sonnenwahrscheinlichkeit von etwa 60 %? Stell dir nun vor, die Simulation des Wetterverlaufs wird 100 Mal unabhängig wiederholt, also R = 100. In wie vielen der 100 Verläufe würdest du ungefähr erwarten, dass an Tag 10 die Sonne scheint? <span style="color: #707070;">(leicht)</span>
2. Lass **k = 10** fest und erhöhe R auf **10, 100 und 1 000**. Vergleiche Beobachtung (blau) und Vorhersage (orange); lose anschließend eine neue Versuchsreihe aus. Muss die Abweichung zwischen blau und orange bei jeder Vergrößerung von R kleiner werden? <span style="color: #707070;">(leicht)</span>
3. Vergleiche **k = 1, 10 und 60** sowie beide Starts. Warum kann die blaue Kurve oben bei großem R nahe an der Vorhersage liegen, obwohl das Wetter unten im einzelnen Verlauf weiterhin wechselt? <span style="color: #707070;">(schwerer)</span>

Die Regler verwenden dieselben Daten: Mehr Tage verlängern die Verläufe, mehr Wiederholungen ergänzen weitere unabhängige Verläufe. Gezählt wird hier **über Verläufe am selben Tag**, nicht über die Tage eines einzelnen Verlaufs.


In [3]:
station_2 = station_wetterverlauf()

<br><br>

## 3 · Wie oft scheint die Sonne? (Skriptaufgabe 1.37.2)

Wir bleiben bei **a = 80 %, b = 30 %**. Vermute vor dem Start, welcher Sonnenanteil sich in einem langen Wetterverlauf einstellen könnte.

Die Grafik zeigt den Anteil sonniger Tage unter den Tagen **1 bis n**; der Starttag 0 zählt nicht mit. Die waagerechte Achse ist logarithmisch: Gleiche Abstände entsprechen gleichen Faktoren.

1. **Vergrößere N** von 10 über 100 und 1 000 bis 100 000. Notiere bei **100 000 Schritten** die Anteile für **Sonne und Regen**. Der Regler verlängert denselben Verlauf. Vergleiche dann mit einem neu ausgelosten Verlauf und einem anderen Startwetter. <span style="color: #707070;">(leicht)</span>
2. Stabilisiert sich der Anteil? Muss jede zusätzliche Beobachtung ihn näher an seinen langfristigen Wert bringen? <span style="color: #707070;">(leicht)</span>
3. Was unterscheidet diesen Zeitanteil von der Sonnenwahrscheinlichkeit an einem einzelnen Tag in Station 1? <span style="color: #707070;">(schwerer)</span>


In [4]:
station_3 = station_zeitanteile()

<br><br>

## 4 · Wie kann man eine Übergangsmatrix schätzen?

Aus einem Wetterverlauf schätzen wir $a=P(S\mid S)$ und $b=P(S\mid R)$ durch

$$\widehat a_n=\frac{\text{Anzahl der Übergänge S → S}}{\text{Anzahl aller Übergänge, die in S starten}},\qquad
\widehat b_n=\frac{\text{Anzahl der Übergänge R → S}}{\text{Anzahl aller Übergänge, die in R starten}}.$$

Gezählt werden jeweils die ersten **n Übergänge**. Die Grafik zeigt oben die laufenden Schätzungen neben den Modellwerten **a = 0,8 und b = 0,3**. Unten siehst du die **vorzeichenbehafteten Fehler in Prozentpunkten**, vergrößert für $n$ von ungefähr $N/10$ bis $N$. Beide Zeitachsen sind logarithmisch; die untere y-Achse passt sich an, damit kleine Abweichungen sichtbar bleiben. Die Tabelle unter der Grafik gehört zum aktuell gewählten $N$.

1. **Vergleiche N = 100, 1 000 und 10 000.** Liegen die Schätzungen jeweils über oder unter dem Modellwert? Welche ist bei diesen drei Werten von $N$ näher am richtigen Wert, die orangene Kurve ($\widehat a_n$) oder die blaue Kurve ($\widehat b_n$)? Nutze für die kleinen Unterschiede den unteren Plot und die Tabelle. <span style="color: #707070;">(leicht)</span>
2. **Stelle N = 10 000 ein.** Finde im Fehlerplot einen Abschnitt, in dem sich eine Schätzung vom richtigen Wert entfernt. Warum widerspricht das nicht der Konvergenz? Lose neue Daten aus: Bleibt immer dieselbe der beiden Kurven (orange bzw. blau) genauer? <span style="color: #707070;">(schwerer)</span>
3. **Übergangswahrscheinlichkeiten von Hand schätzen:** An 15 aufeinanderfolgenden Tagen wurde folgende Wetterfolge beobachtet:

   **S – S – R – R – S – R – S – S – S – R – R – R – S – R – S**

   Zähle die Übergänge zwischen jeweils zwei aufeinanderfolgenden Tagen, getrennt nach **S → S, S → R, R → S und R → R**. Berechne daraus $\widehat a$ und $\widehat b$ als Brüche. <span style="color: #707070;">(leicht)</span>

4. Stelle aus deinen Schätzungen die vollständige Übergangsmatrix auf (Reihenfolge S, R; Spalten: heute, Zeilen: morgen). <span style="color: #707070;">(schwerer)</span>


In [5]:
station_4 = station_schaetzlabor()

<br><br>

## Kleine Codeaufgabe · Die Vorhersage selbst berechnen (optional)

Jetzt bearbeitest du wenige Zeilen selbst. Die Matrix M verwendet wieder die Reihenfolge **Sonne, Regen**; `p = [1, 0]` bedeutet einen sicher sonnigen Start.

1. **Ersetze `None` durch die Rechnung für die Verteilung einen Tag später.** Verwende M und p; in Python steht `@` für Matrix-Vektor-Multiplikation. Die Schleife wiederholt deine Rechnung für jeden Tag, `p = p_neu` übernimmt das Ergebnis für den nächsten Schritt. <span style="color: #707070;">(schwerer)</span>
2. Führe die Zelle aus. Ändere danach `tage` oder starte mit `p = np.array([0.0, 1.0])`. <span style="color: #707070;">(leicht)</span>
3. Vergleiche mit dem Modelllabor bei 80 % und 30 %. Die Ausgabe nennt pro Tag die Wahrscheinlichkeit für Sonne und Regen – nicht das Wetter eines einzelnen Versuchs. <span style="color: #707070;">(leicht)</span>


In [6]:
M = np.array([[0.8, 0.3],
              [0.2, 0.7]])
p = np.array([1.0, 0.0])
tage = 5

for tag in range(1, tage + 1):
    p_neu = None  # Ergänze hier deine Rechnung für den nächsten Tag.
    if p_neu is None:
        print("Ersetze zuerst None durch die passende Rechnung.")
        break
    p = p_neu
    print(f"Tag {tag}: Sonne {p[0]:.3f}, Regen {p[1]:.3f}")

Ersetze zuerst None durch die passende Rechnung.
